# 01 Data exploration

This notebook answers one question: what do the four daily price series in the
frozen panel actually look like, and which of their properties should shape the
modelling choices made later.

It reads `data/interim/panel_1d.parquet` and nothing else. It does not download
data and it does not train anything.

**Run this notebook from the repository root**, so that `from src...` imports
resolve. If it is launched from inside `notebooks/`, the first code cell walks
one directory up to find the repository root.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import norm

from src import schema
from src.data.build_panel import load_panel

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print("repository root:", ROOT)

## The panel

The panel is long format with one row per asset and date, carrying an
`(asset, date)` MultiIndex. Every later notebook and every model reads this same
file, so the shape reported here is the shape everything downstream sees.

In [ ]:
panel = load_panel(ROOT / "data" / "interim" / "panel_1d.parquet")
schema.check_panel_index(panel)

assets = list(panel.index.get_level_values(schema.ASSET).unique())
dates = panel.index.get_level_values(schema.DATE)

print("rows:", len(panel))
print("assets:", assets)
print("date range:", dates.min().date(), "to", dates.max().date())
panel.head()

## Coverage per asset

The four assets were not listed at the same time, so the panel is unbalanced.
This matters for anything computed across assets at once: a correlation between
BTC and SOL is estimated on a much shorter sample than a correlation between BTC
and ETH, and a model trained on pooled rows sees far more BTC history than SOL
history.

The `missing_days` column is the difference between the calendar span and the
number of rows. Crypto trades every day, so a non-zero value there is a genuine
gap in the exchange archive rather than a weekend.

In [ ]:
flat = panel.reset_index()
coverage = flat.groupby(schema.ASSET, observed=True).agg(
    rows=(schema.DATE, "size"),
    first_date=(schema.DATE, "min"),
    last_date=(schema.DATE, "max"),
)
coverage["calendar_days"] = (coverage["last_date"] - coverage["first_date"]).dt.days + 1
coverage["missing_days"] = coverage["calendar_days"] - coverage["rows"]
coverage["first_date"] = coverage["first_date"].dt.date
coverage["last_date"] = coverage["last_date"].dt.date
coverage

## Price levels

Prices are plotted on a log scale and each series is divided by its own first
observation, so every asset starts at 1.0 and equal vertical distances mean
equal percentage moves. Without the log scale the 2021 and 2024 moves compress
everything before them into a flat line.

The normalisation is per asset rather than per calendar date, because the assets
start on different dates. SOL starting at 1.0 in 2020 is not a claim that it was
worth the same as BTC then.

In [ ]:
close_wide = panel[schema.CLOSE].unstack(level=schema.ASSET).sort_index()
close_wide = close_wide[list(assets)]

normalised = close_wide.apply(lambda s: s / s.dropna().iloc[0])

figure, axis = plt.subplots(figsize=(12, 6))
for asset in assets:
    axis.plot(normalised.index, normalised[asset], linewidth=1.1, label=asset)
axis.set_yscale("log")
axis.set_ylabel("close, rebased to 1 at the first observation")
axis.set_xlabel("date")
axis.set_title("Log price paths, each asset rebased to its own start")
axis.legend()
figure.tight_layout()
plt.show()

## Daily log returns

Returns are close to close in UTC and taken in logs, matching
`src.data.targets.build_targets`, so the series examined here is the same object
the models are asked to forecast.

Each histogram has a normal density fitted to that asset's own mean and standard
deviation drawn over it. The vertical axis is logarithmic, which is what makes
the tails readable: on a linear axis the tail bins are invisible and the two
curves look almost identical. The gap between the histogram and the fitted curve
in the far tails is the fat tail behaviour, and the gap at the centre is the
excess peak that comes with it.

In [ ]:
returns = np.log(close_wide).diff()

figure, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
for axis, asset in zip(axes.ravel(), assets):
    series = returns[asset].dropna()
    axis.hist(series, bins=150, density=True, color="0.65", edgecolor="none")
    grid = np.linspace(series.min(), series.max(), 500)
    axis.plot(grid, norm.pdf(grid, series.mean(), series.std()), "r-", linewidth=1.2,
              label="fitted normal")
    axis.set_yscale("log")
    axis.set_title("%s, n = %d" % (asset, len(series)))
    axis.set_xlabel("daily log return")
    axis.legend(fontsize=8)
figure.suptitle("Daily log return distributions against a fitted normal, log density axis")
figure.tight_layout()
plt.show()

## Return moments

Annualised volatility uses 365 days because crypto trades every calendar day, so
the 252 day convention borrowed from equities would understate it.

Excess kurtosis is reported, so a normal distribution would score zero. Values in
the high single digits or above mean the largest daily moves are far more
frequent than a normal law allows, which is the numerical version of the tail gap
in the plot above. Any model or evaluation that assumes normal errors is making
an assumption this table does not support.

In [ ]:
rows = []
for asset in assets:
    series = returns[asset].dropna()
    rows.append({
        "asset": asset,
        "n": len(series),
        "mean": series.mean(),
        "std": series.std(),
        "ann_vol_365": series.std() * np.sqrt(365.0),
        "skew": series.skew(),
        "excess_kurtosis": series.kurt(),
        "min": series.min(),
        "max": series.max(),
    })
moments = pd.DataFrame(rows).set_index("asset")
moments.round(4)

## Correlation across assets

The static matrix below is computed pairwise, so each cell uses whatever sample
the two assets have in common. The BTC and SOL cell rests on the SOL history
alone while the BTC and ETH cell rests on the full panel, and the two numbers are
therefore not measured over the same market conditions. The second panel of the
figure reports the number of overlapping observations behind each cell so that
this is visible rather than assumed.

A single correlation number is only useful if the relationship is stable. The
figure after it recomputes each pair on a rolling 90 day window. If those lines
move over a wide range then any model that treats cross asset relationships as
fixed is fitting an average of several different regimes.

In [ ]:
static_corr = returns.corr()
present = returns.notna().astype(int)
overlap = present.T.dot(present)

figure, axes = plt.subplots(1, 2, figsize=(12, 4.6))
image = axes[0].imshow(static_corr.to_numpy(), vmin=0.0, vmax=1.0, cmap="viridis")
axes[0].set_xticks(range(len(assets)), assets)
axes[0].set_yticks(range(len(assets)), assets)
axes[0].set_title("Correlation of daily log returns")
axes[0].grid(False)
for i in range(len(assets)):
    for j in range(len(assets)):
        axes[0].text(j, i, "%.2f" % static_corr.iloc[i, j], ha="center", va="center",
                     color="white", fontsize=9)
figure.colorbar(image, ax=axes[0], fraction=0.046)

image2 = axes[1].imshow(overlap.to_numpy(), cmap="magma")
axes[1].set_xticks(range(len(assets)), assets)
axes[1].set_yticks(range(len(assets)), assets)
axes[1].set_title("Overlapping observations behind each cell")
axes[1].grid(False)
for i in range(len(assets)):
    for j in range(len(assets)):
        axes[1].text(j, i, "%d" % overlap.iloc[i, j], ha="center", va="center",
                     color="white", fontsize=8)
figure.colorbar(image2, ax=axes[1], fraction=0.046)
figure.tight_layout()
plt.show()

static_corr.round(3)

In [ ]:
pairs = [(a, b) for i, a in enumerate(assets) for b in list(assets)[i + 1:]]

rolling = pd.DataFrame(index=returns.index)
for left, right in pairs:
    rolling["%s-%s" % (left, right)] = (
        returns[left].rolling(90, min_periods=60).corr(returns[right])
    )

figure, axis = plt.subplots(figsize=(12, 5.5))
for column in rolling.columns:
    axis.plot(rolling.index, rolling[column], linewidth=1.0, label=column)
axis.axhline(0.0, color="black", linewidth=0.8)
axis.set_ylabel("rolling 90 day correlation")
axis.set_xlabel("date")
axis.set_title("Pairwise correlation of daily log returns is not constant")
axis.legend(fontsize=8, ncol=3)
figure.tight_layout()
plt.show()

rolling.describe().round(3)

## Volatility clustering

Large moves arrive next to other large moves. The absolute return series below
shows this directly: activity is concentrated in bursts rather than spread evenly
through time, so the size of tomorrow's move is partly predictable from today
even when its sign is not.

The figure after it puts the two autocorrelation functions on the same axes. The
autocorrelation of the raw returns answers the question "does the market repeat
its direction". A few of its lags clear the confidence band, but the largest of
them is around 0.05, which is a correlation small enough that a linear rule built
on it would be swamped by trading costs. The autocorrelation of the absolute
returns answers the question "does the market repeat its activity level", and it
is several times larger and stays positive for weeks.

Read the comparison in magnitudes rather than in whether a bar crosses a line.
With more than three thousand observations the band is narrow enough that
economically negligible dependence becomes statistically visible, so crossing it
is not on its own a reason to expect a forecast.

In [ ]:
figure, axes = plt.subplots(len(assets), 1, figsize=(12, 8), sharex=True)
for axis, asset in zip(axes, assets):
    series = returns[asset].dropna()
    axis.plot(series.index, series.abs(), linewidth=0.6, color="0.25")
    axis.set_ylabel(asset)
axes[0].set_title("Absolute daily log return, activity arrives in bursts")
axes[-1].set_xlabel("date")
figure.tight_layout()
plt.show()

In [ ]:
from statsmodels.tsa.stattools import acf

MAX_LAG = 40

figure, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True)
for axis, asset in zip(axes.ravel(), assets):
    series = returns[asset].dropna()
    lags = np.arange(1, MAX_LAG + 1)
    acf_return = acf(series, nlags=MAX_LAG, fft=True)[1:]
    acf_absolute = acf(series.abs(), nlags=MAX_LAG, fft=True)[1:]
    band = 1.96 / np.sqrt(len(series))

    axis.plot(lags, acf_return, "o-", markersize=3, linewidth=1.0, label="returns")
    axis.plot(lags, acf_absolute, "s-", markersize=3, linewidth=1.0,
              label="absolute returns")
    axis.axhline(band, color="black", linestyle=":", linewidth=0.8)
    axis.axhline(-band, color="black", linestyle=":", linewidth=0.8)
    axis.axhline(0.0, color="black", linewidth=0.6)
    axis.set_title("%s, dotted band is 1.96 / sqrt(n)" % asset)
    axis.set_xlabel("lag in days")
    axis.legend(fontsize=8)
figure.suptitle("Returns carry little linear memory, their magnitudes carry a lot")
figure.tight_layout()
plt.show()

## What this implies for the rest of the project

1. Returns are the modelling unit, not prices. The price paths above trend over
   several orders of magnitude, and a model fitted to levels would score well by
   repeating yesterday's close. Notebook 02 tests this formally.
2. Direction is a hard problem. The largest return autocorrelation across the
   first 40 lags is around 0.05, so whatever linear signal exists is far too
   small to build a rule on, and a directional model has to be judged against the
   per fold base rate rather than against 50 percent.
3. Volatility is a comparatively tractable problem. The absolute return
   autocorrelations are large and persistent, which is why the volatility target
   exists and why a persistence baseline on volatility is a demanding one.
4. Cross asset relationships move. Correlations estimated on the full sample do
   not describe any particular year, so the evaluation has to report per fold
   results rather than a pooled average.